## Export Variant Video Library

- pull these functions into the main library later

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
from dask.distributed import wait

import copy
import h5py
import cv2
import skimage as sk

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

In [ ]:
# def lookups_to_output_arr(headpath, x, dir_file_name = '/kymograph/kymograph_'):
#     for i in range(len(x)):
#         with h5py.File(headpath + dir_file_name + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
#             if i == 0:
#                 output_arr_dict = {channel:[] for channel in list(infile.keys())}
#             for channel in list(infile.keys()):
#                 if x.iloc[i]["lane orientation"] == "top":
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
#                 else:
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
#     for channel in list(output_arr_dict.keys()):
#         output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
#     return output_arr_dict

def lookups_to_output_arr(headpath, x, dir_file_name = '/kymograph/kymograph_', is_seg = False):
    for i in range(len(x)):
        with h5py.File(headpath + dir_file_name + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
            if i == 0:
                output_arr_dict = {channel:[] for channel in list(infile.keys())}
            for channel in list(infile.keys()):
                if x.iloc[i]["lane orientation"] == "top":
                    output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
                else:
                    # LUIS: ADDITION MADE TO BE CONSIDERATE WITH PHOTOSENSITIVE VIEWERS
                    img = infile[channel][x.iloc[i]["File Trench Index"]]
                    # if is_seg: # Relabel regions if 
                    #     img2 = np.zeros(img.shape)
                    #     inds = np.unique(img)
                    #     inds = inds[np.nonzero(inds)[0]]
                    #     for ind in inds:
                    #         img2[img == ind] = len(inds) - ind + 1
                    #     img = img2
                    output_arr_dict[channel].append(img)
    for channel in list(output_arr_dict.keys()):
        output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
    return output_arr_dict

def export_avi(output_arr,outpath,min_intensity = 0,max_intensity = 10000, fps=10, is_seg=False):
    
    if is_seg: # do not scale
        output_arr_scaled = output_arr
    else: 
        output_arr_scaled = sk.exposure.rescale_intensity(output_arr,in_range=(min_intensity,max_intensity))
    
    # fourcc = cv2.VideoWriter_fourcc(*'MJPG') # DANIEL
    fourcc = cv2.VideoWriter_fourcc(*'H264') # LUIS
    out = cv2.VideoWriter(outpath,fourcc, fps, (output_arr_scaled.shape[2],output_arr_scaled.shape[1]))
    
    if is_seg: # Label to RGB, then make 8 bit int
        for t in range(output_arr_scaled.shape[0]):
            out.write(sk.util.img_as_ubyte(sk.color.label2rgb(output_arr_scaled[t])))
    else: # Grayscale to RGB
        for t in range(output_arr_scaled.shape[0]):
            out.write(sk.color.gray2rgb(sk.util.img_as_ubyte(output_arr_scaled[t])))
    out.release()
    
def lookup_to_avi(headpath_list,outpath,channel_params_dict,x,dir_file_name = '/kymograph/kymograph_', is_seg=False):
    tr.writedir(outpath, overwrite=False)
    
    output_arr_dict_list = []
    for i,headpath in enumerate(headpath_list):
        filtered_x = x[x["Experiment #"]==i]
        if len(filtered_x)>0:
            output_arr_dict = lookups_to_output_arr(headpath,filtered_x,dir_file_name,is_seg=is_seg)
            output_arr_dict_list.append(output_arr_dict)
        
    merged_output_arr_dict = {}
    for channel in list(output_arr_dict.keys()):
        merged_output_arr_dict[channel] = np.concatenate([output_arr_dict[channel] for output_arr_dict in output_arr_dict_list], axis=2)
    del output_arr_dict_list
        
    for channel in list(merged_output_arr_dict.keys()):
        selected_channel_params = channel_params_dict[channel]
        min_intensity,max_intensity,fps = selected_channel_params
        tr.writedir(outpath + "/" + channel, overwrite=False)
        # export_avi(merged_output_arr_dict[channel],outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".avi",\
        #            min_intensity = min_intensity, max_intensity = max_intensity, fps=fps) # DANIEL
        export_avi(merged_output_arr_dict[channel],outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".mp4",\
                   min_intensity = min_intensity, max_intensity = max_intensity, fps=fps,  is_seg=is_seg) # LUIS
        
    return True
# Added by Luis for segmentation of cells and nucleoids
# TODO: It is very hacky, incorporate with Daniel's function. For instance, the scaling parameters do not need to be used, the path layout is different, there is only the 'data' channel.

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=100,
    n_workers_min=30,
    memory="16GB",
    working_directory="/home/lag36/scratch/lag36/dask/",
)
dask_controller.startdask()
dask_controller.displaydashboard()

In [ ]:
output_phenotype_kmyo_df_path_list = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Final_Barcodes_df_Run_2/"]
output_phenotype_kmyo_df_list = [dd.read_parquet(item,engine="pyarrow",calculate_divisions=True) for item in output_phenotype_kmyo_df_path_list]

# trench_to_lookup_df_list = []
variant_to_trench_df_list = []

for i in range(len(output_phenotype_kmyo_df_list)):
    trench_to_lookup_df = output_phenotype_kmyo_df_list[i].set_index("phenotype trenchid",sorted=True).groupby('phenotype trenchid').apply(lambda x: x.iloc[0])
    trench_to_lookup_df = trench_to_lookup_df[['File Index','File Trench Index','lane orientation','oDEPool7_id']].compute()
    trench_to_lookup_df["oDEPool7_id Temp"] = copy.deepcopy(trench_to_lookup_df["oDEPool7_id"])
    trench_to_lookup_df["Experiment #"] = i
    variant_to_trench_df = trench_to_lookup_df.reset_index().set_index("oDEPool7_id")
    variant_to_trench_df_list.append(variant_to_trench_df)
    
variant_to_trench_df = pd.concat(variant_to_trench_df_list,axis=0).sort_index()
variant_to_trench_dd = dd.from_pandas(variant_to_trench_df,npartitions=200).persist()

In [ ]:
# dir_file_name_fl = '/kymograph/kymograph_'
# dir_file_name_cell_seg = '/fluorsegmentation/segmentation_'
# dir_file_name_nuc_seg = '/nucleoidsegmentation/segmentation_'
# dir_file_name = dir_file_name_fl

### Generate Videos from Intensity channels

In [ ]:
headpath_list = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division"]
output_path = '/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/'
channel_params_dict = {"YFP":(0,6000,10), "mCherry":(0,6000,10)}
dir_file_name = '/kymograph/kymograph_'
is_seg = False

tr.writedir(output_path, overwrite=True)
variant_to_trench_dd.groupby("oDEPool7_id").apply(lambda x: lookup_to_avi(headpath_list,
                                                                          output_path,
                                                                          channel_params_dict,
                                                                          x,
                                                                          dir_file_name,
                                                                          is_seg),meta=bool).compute()

### Generate Videos from Segmentation Kymographs

In [ ]:
headpath_list = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division"]
output_path = '/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/Nucleoid_Segmentation/'
channel_params_dict = {"data":(0,15,10)}
dir_file_name = '/nucleoidsegmentation/segmentation_'
is_seg = True

tr.writedir(output_path, overwrite=True)
variant_to_trench_dd.groupby("oDEPool7_id").apply(lambda x: lookup_to_avi(headpath_list,
                                                                          output_path,
                                                                          channel_params_dict,
                                                                          x,
                                                                          dir_file_name,
                                                                          is_seg),meta=bool).compute()

### Generate Videos from Segmentation Kymographs

In [ ]:
headpath_list = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division"]
output_path = '/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/Cell_Segmentation/'
dir_file_name = '/fluorsegmentation/segmentation_'
channel_params_dict = {"data":(0,15,10)}
is_seg = True

tr.writedir(output_path, overwrite=True)
variant_to_trench_dd.groupby("oDEPool7_id").apply(lambda x: lookup_to_avi(headpath_list,
                                                                          output_path,
                                                                          channel_params_dict,
                                                                          x,
                                                                          dir_file_name,
                                                                          is_seg),meta=bool).compute()

In [ ]:
# Test
from IPython.display import Video
Video("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/Nucleoid_Segmentation/data/1027.mp4", embed=True)

## Merging different channels into a single video (Luis)
This tool merges exported videos from `4_Video_Export.ipynb` into a single video of all imaging channels combined. It used `videos_grid.py`, which should be included in the same directory

In [ ]:
import videos_grid
from IPython.display import Video

import os

Please set the following paths and channels. Videos should be stored in `headpath_videos/channels[i]` for all channels in `channels`

In [ ]:
headpath_videos = '/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/'
output_path = '/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/Merged_with_Seg/' # Make sure the directory exists
channels = ["mCherry", "Nucleoid_Segmentation/data/", "YFP", "Cell_Segmentation/data/"]

mch_files = os.listdir(headpath_videos+'/'+channels[0])
nuc_seg_files = os.listdir(headpath_videos+'/'+channels[1])
yfp_files = os.listdir(headpath_videos+'/'+channels[2])
cell_seg_files = os.listdir(headpath_videos+'/'+channels[3])

mch_ids = sorted([file.split('.')[0] for file in mch_files])
nuc_seg_ids = sorted([file.split('.')[0] for file in nuc_seg_files])
yfp_ids = sorted([file.split('.')[0] for file in yfp_files])
cell_seg_ids = sorted([file.split('.')[0] for file in cell_seg_files])

print([x == y for x in [mch_ids, nuc_seg_ids, yfp_ids, cell_seg_ids] for y in [mch_ids, nuc_seg_ids, yfp_ids, cell_seg_ids]])

In [ ]:
# Function to merge videos
def merge_video(oDEPool7_id,
                     headpath_videos = headpath_videos,
                     output_path = output_path,
                     channels = channels):
    videos_to_merge = [headpath_videos + '/' + channel + '/' + oDEPool7_id + '.mp4' for channel in channels]
    videos_grid.merge_videos(videos_to_merge,
                             output_path+'/'+ oDEPool7_id+'.mp4',
                             grid_size=(4, 1),  # 2 row, 2 cols
                             max_frames=61)  # merge first 100 frames per video
    return oDEPool7_id

In [ ]:
# # Run locally for testing
# for oDEPool7_id in mch_ids[:1]:
#     videos_to_merge = [headpath_videos + '/' + channel + '/' + oDEPool7_id + '.mp4' for channel in channels]
#     videos_grid.merge_videos(
#     videos_to_merge,
#     output_path+'/'+ oDEPool7_id+'.mp4',
#     grid_size=(2, 1),  # 2 row, 2 cols
#     max_frames=61)  # merge first 100 frames per video
#     # break

In [ ]:
# Compute scheduler='threads'

In [ ]:
# Run in Dask
tr.writedir(output_path+'/', overwrite=True)
futures_dict = {}
for oDEPool7_id in mch_ids:
    futures_dict['oDEPool7_id: '+str(oDEPool7_id)] = dask_controller.daskclient.submit(merge_video, oDEPool7_id)

# END OF SCRIPT

### Testing

In [ ]:
output_path+'/'+ oDEPool7_id+'.mp4'

In [ ]:
dask_controller.shutdown()

In [ ]:
## TEST
# TEST
df27 = variant_to_trench_df.groupby("oDEPool7_id").get_group(27)

##
headpath_list = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division"]
output_path = '/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/MP4_Library/Cell_Segmentation/'
dir_file_name = '/fluorsegmentation/segmentation_'
channel_params_dict = {"data":(0,15,10)}
is_seg = True

tr.writedir(output_path, overwrite=True)

##
lookup_to_avi(headpath_list,
              output_path,
              channel_params_dict,
              df27,
              dir_file_name,
              is_seg)

##
idx = 0
ch = 'data'
t_inds = np.arange(10)
with h5py.File("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/nucleoidsegmentation/segmentation_0.hdf5", "r") as infile:  
    img = infile[ch][idx][t_inds[0]]
    
#     if len(t_inds) > 1:
#         for t_ind in t_inds[1:]:
#             img_ind = infile[ch][idx][t_ind]
#             img = np.concatenate((img, img_ind), axis = 1)

#     fig, ax = plt.subplots();
#     ax.imshow(img, cmap='jet');

plt.imshow(img, cmap = 'jet')

img2 = np.zeros(img.shape)
inds = np.unique(img)
print(inds)
inds = inds[np.nonzero(inds)[0]]
for ind in inds:
    img2[img == ind] = len(inds) - ind + 1

# Old functions before Adding segmentation

In [ ]:
# import h5py
# import cv2
# import skimage as sk
# from skimage.color import gray2rgb
# import copy

# def lookups_to_output_arr(headpath,x):
#     for i in range(len(x)):
#         with h5py.File(headpath + "/kymograph/kymograph_" + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
#             if i == 0:
#                 output_arr_dict = {channel:[] for channel in list(infile.keys())}
#             for channel in list(infile.keys()):
#                 if x.iloc[i]["lane orientation"] == "top":
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
#                 else:
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
#     for channel in list(output_arr_dict.keys()):
#         output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
#     return output_arr_dict

# def export_avi(output_arr,outpath,min_intensity = 0,max_intensity = 10000, fps=10):
#     output_arr_scaled = sk.exposure.rescale_intensity(output_arr,in_range=(min_intensity,max_intensity))
#     # fourcc = cv2.VideoWriter_fourcc(*'MJPG') # DANIEL
#     fourcc = cv2.VideoWriter_fourcc(*'H264') # LUIS
    
#     out = cv2.VideoWriter(outpath,fourcc, fps, (output_arr_scaled.shape[2],output_arr_scaled.shape[1]))
#     for t in range(output_arr_scaled.shape[0]):
#         out.write(sk.color.gray2rgb(sk.util.img_as_ubyte(output_arr_scaled[t])))
#     out.release()
    
# def lookup_to_avi(headpath_list,outpath,channel_params_dict,x):
#     tr.writedir(outpath, overwrite=False)
    
#     output_arr_dict_list = []
#     for i,headpath in enumerate(headpath_list):
#         filtered_x = x[x["Experiment #"]==i]
#         if len(filtered_x)>0:
#             output_arr_dict = lookups_to_output_arr(headpath,filtered_x)
#             output_arr_dict_list.append(output_arr_dict)
        
#     merged_output_arr_dict = {}
#     for channel in list(output_arr_dict.keys()):
#         merged_output_arr_dict[channel] = np.concatenate([output_arr_dict[channel] for output_arr_dict in output_arr_dict_list], axis=2)
#     del output_arr_dict_list
        
#     for channel in list(merged_output_arr_dict.keys()):
#         selected_channel_params = channel_params_dict[channel]
#         min_intensity,max_intensity,fps = selected_channel_params
#         tr.writedir(outpath + "/" + channel, overwrite=False)
#         # export_avi(merged_output_arr_dict[channel],outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".avi",\
#         #            min_intensity = min_intensity, max_intensity = max_intensity, fps=fps) # DANIEL
#         export_avi(merged_output_arr_dict[channel],outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".mp4",\
#                    min_intensity = min_intensity, max_intensity = max_intensity, fps=fps) # LUIS
        
#     return True

# # Added by Luis for segmentation of cells and nucleoids
# # TODO: It is very hacky, incorporate with Daniel's function. For instance, the scaling parameters do not need to be used, the path layout is different, there is only the 'data' channel.
# def lookups_to_output_arr_cell_seg(headpath,x):
#     for i in range(len(x)):
#         with h5py.File(headpath + "/fluorsegmentation/segmentation_" + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
#             if i == 0:
#                 output_arr_dict = {channel:[] for channel in list(infile.keys())}
#             for channel in list(infile.keys()):
#                 if x.iloc[i]["lane orientation"] == "top":
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
#                 else:
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
#     for channel in list(output_arr_dict.keys()):
#         output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
#     return output_arr_dict

# def lookups_to_output_arr_nuc_seg(headpath,x):
#     for i in range(len(x)):
#         with h5py.File(headpath + "/nucleoidsegmentation/segmentation_" + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
#             if i == 0:
#                 output_arr_dict = {channel:[] for channel in list(infile.keys())}
#             for channel in list(infile.keys()):
#                 if x.iloc[i]["lane orientation"] == "top":
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
#                 else:
#                     output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
#     for channel in list(output_arr_dict.keys()):
#         output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
#     return output_arr_dict

# def export_avi_seg(output_arr,outpath,min_intensity = 0,max_intensity = 10000, fps=10):
#     fourcc = cv2.VideoWriter_fourcc(*'H264') # LUIS
    
#     out = cv2.VideoWriter(outpath,fourcc, fps, (output_arr.shape[2],output_arr.shape[1]))
#     for t in range(output_arr.shape[0]):
#         out.write(sk.util.img_as_ubyte(sk.color.label2rgb(output_arr[t])))
#     out.release()

### Interactive Variant Overlay

In [ ]:
# def linked_kymograph_for_gene_table(xrstack,wrapped_xrstack,df,select_gene,select_trenchid,select_unpacked_trenchid=None,trenchid_column='trenchid',
#                                     y_scale=3,x_window_size=300,rescale_videos=True,fps=10,dpi=200):
#     ### stream must return trenchid value
#     #### df must have trenchid lookups
#     width,height = xrstack.shape[3],int(xrstack.shape[2]*y_scale)
#     x_window_scale = x_window_size/xrstack.shape[3]
#     x_size = int(xrstack.shape[3]*(y_scale*x_window_scale))

#     df = df.reset_index(drop=False).set_index("Gene",drop=False).sort_index()
#     df.index = df.index.rename("Gene Index")
#     df_genes_only = df[["Gene"]].groupby("Gene").apply(lambda x: x.iloc[0])
#     dataset = hv.Dataset(df)
#     gene_dataset = hv.Dataset(df_genes_only)

#     if select_unpacked_trenchid != None:
#         def select_pt_load_image(channel,select_gene_index,trenchid_index,unpack_trenchid_index,width=width,height=height):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             double_filtered_dataset = hv.Dataset({"trenchid": filtered_dataset.iloc[trenchid_index][trenchid_column][0]},['trenchid'])
#             trenchid = double_filtered_dataset.iloc[unpack_trenchid_index]["trenchid"][0]
#             arr = xrstack.loc[channel,trenchid].compute(scheduler="threads").values
#             return hv.Image(arr,bounds=(0,0,width,height))

#         def print_trenchid(select_gene_index,trenchid_index,unpack_trenchid_index):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             double_filtered_dataset = hv.Dataset({"trenchid": filtered_dataset.iloc[trenchid_index][trenchid_column][0]},['trenchid'])
#             trenchid = double_filtered_dataset.iloc[unpack_trenchid_index]["trenchid"][0]
#             return hv.Text(3.,20.,str(trenchid),fontsize=30)

#     else:
#         def select_pt_load_image(channel,select_gene_index,trenchid_index,width=width,height=height):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             trenchid = filtered_dataset.iloc[trenchid_index][trenchid_column][0]
#             arr = xrstack.loc[channel,trenchid].compute(scheduler="threads").values
#             return hv.Image(arr,bounds=(0,0,width,height))

#         def print_trenchid(select_gene_index,trenchid_index):
#             gene_name = gene_dataset.iloc[select_gene_index]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             trenchid = filtered_dataset.iloc[trenchid_index][trenchid_column][0]
#             return hv.Text(3.,20.,str(trenchid),fontsize=30)

#     def set_bounds(fig, element, y_dim=height, x_dim=width, x_window_size=x_window_size):
#         sy = y_dim-0.5
#         sx = x_dim-0.5
        
#         x_zoom_pos = (fig.state.x_range.start,fig.state.x_range.end)
#         xdata = element.dataset.data['x']-0.5
#         xdata_range = (xdata.min(), xdata.max()+1.)
#         init_test = x_zoom_pos==xdata_range
        
#         fig.state.x_range.bounds = (0, sx)
#         fig.state.y_range.bounds = (-0.5, sy)

#         if init_test:
#             fig.state.x_range.start = 0
#             fig.state.x_range.end = x_window_size
#         else:
#             fig.state.x_range.start = x_zoom_pos[0]
#             fig.state.x_range.end = x_zoom_pos[0]+x_window_size
            
#         fig.state.y_range.start = 0
#         fig.state.y_range.end = y_dim
#         fig.state.x_range.reset_start = 0
#         fig.state.x_range.reset_end = x_window_size
#         fig.state.y_range.reset_start = 0
#         fig.state.y_range.reset_end = y_dim

#     if select_unpacked_trenchid != None:
#         image_stack = hv.DynamicMap(select_pt_load_image, kdims=['Channel'], streams=[select_gene,select_trenchid,select_unpacked_trenchid])
#         trenchid_display = hv.DynamicMap(print_trenchid, streams=[select_gene,select_trenchid,select_unpacked_trenchid])
#     else:
#         image_stack = hv.DynamicMap(select_pt_load_image, kdims=['Channel'], streams=[select_gene,select_trenchid])
#         trenchid_display = hv.DynamicMap(print_trenchid, streams=[select_gene,select_trenchid])

#     kymograph_display = image_stack.opts(plot={'Image': dict(colorbar=True, tools=['hover'],hooks=[set_bounds],aspect='equal'),})
#     kymograph_display = kymograph_display.opts(cmap='Greys_r',height=height,width=x_size)

#     kymograph_display = kymograph_display.redim.range(trenchid=(0,xrstack.shape[1]))
#     kymograph_display = kymograph_display.redim.values(Channel=xrstack.coords["Channel"].values.tolist())

#     if select_unpacked_trenchid != None:

#         def save_video(event):
#             hv.extension('matplotlib')
#             channel = kymograph_display.dimension_values("Channel")[0]
#             gene_name = gene_dataset.iloc[select_gene.index[0]]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             double_filtered_dataset = hv.Dataset({"trenchid": filtered_dataset.iloc[select_trenchid.index[0]][trenchid_column][0]},['trenchid'])
#             trenchid = double_filtered_dataset.iloc[select_unpacked_trenchid.index[0]]["trenchid"][0]
#             arr = wrapped_xrstack.loc[channel,trenchid].values
#             if rescale_videos:
#                 arr = np.array([sk.exposure.rescale_intensity(arr[t]) for t in range(arr.shape[0])])
#             hmap = hv.HoloMap([(t, hv.Raster(arr[t])) for t in range(0,arr.shape[0])], kdims=['t'])
#             hmap = hmap.opts(plot={'Raster': dict(tools=['hover'],aspect='equal',cmap='Greys_r'),})
#             hmap = hmap.opts(cmap='Greys_r')
#             hv.save(hmap, gene_name + '_trenchid=' + str(trenchid) + '.mp4', fmt='mp4', fps=fps, dpi=dpi , backend='matplotlib')
#             hv.extension('bokeh')

#     else:

#         def save_video(event):
#             hv.extension('matplotlib')
#             channel = kymograph_display.dimension_values("Channel")[0]
#             gene_name = gene_dataset.iloc[select_gene.index[0]]["Gene"][0]
#             filtered_dataset = hv.Dataset(df.loc[gene_name:gene_name])
#             trenchid = filtered_dataset.iloc[select_trenchid.index[0]][trenchid_column][0]
#             arr = wrapped_xrstack.loc[channel,trenchid].values
#             if rescale_videos:
#                 arr = np.array([sk.exposure.rescale_intensity(arr[t]) for t in range(arr.shape[0])])
#             hmap = hv.HoloMap([(t, hv.Raster(arr[t])) for t in range(0,arr.shape[0])], kdims=['t'])
#             hmap = hmap.opts(plot={'Raster': dict(tools=['hover'],aspect='equal',cmap='Greys_r'),})
#             hmap = hmap.opts(cmap='Greys_r')
#             hv.save(hmap, gene_name + '_trenchid=' + str(trenchid) + '.mp4', fmt='mp4', fps=fps, dpi=dpi , backend='matplotlib')
#             hv.extension('bokeh')

#     trenchid_display = trenchid_display.opts(text_align="left",text_color="white")

#     output_display = kymograph_display*trenchid_display

#     button = pn.widgets.button.Button(name='Save Video', width=60)
#     button.on_click(save_video)

#     return output_display, button

In [ ]:
output_phenotype_kmyo_df = dd.read_parquet("/home/de64/scratch/de64/sync_folder/2022-07-27_lDE20_Merged_Analysis/2022-07-31_lDE20_Final_Barcodes_df_Merged/",calculate_divisions=True)

In [ ]:
output_phenotype_kmyo_df.index

In [ ]:
headpath = "/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/Growth_Division/"
#note: shutdown dask when doing this...fix bug later
overlay_handle = tr.variant_overlay(headpath,"/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/2022-07-01_lDE23_Final_Barcodes_df/",
                                 display_values_list=["Gene","TargetID","N Mismatch"],persist_data=False)  ##fix this, was improperly made (only initial cellID timepoints)

In [ ]:
overlay_handle.view_overlay(vmin=0,vmax=10000)

### Interactive Variant Table

In [ ]:
gene_table = dd.read_parquet("/home/de64/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1",engine='pyarrow',calculate_divisions=True)
gene_table = gene_table.reset_index().set_index("phenotype trenchid", sorted=True)
gene_table = gene_table.groupby("phenotype trenchid",sort=False).apply(lambda x: x.iloc[0]).reset_index().set_index("FOV Parquet Index")
gene_table_out = gene_table.groupby("sgRNA").apply(lambda x: x.iloc[0])
gene_table_out["phenotype trenchids"] = gene_table.groupby("sgRNA").apply(lambda x: x["phenotype trenchid"].tolist())
gene_table_out = gene_table_out[["Gene",'Target Sequence',"phenotype trenchids","N Mismatch","N Target Sites","Category","Strand",]].compute()

In [ ]:
kymo_xarr = tr.kymo_xarr("/home/de64/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division")
wrapped_kymo_xarr = tr.kymo_xarr("/home/de64/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division",unwrap=False)

In [ ]:
gene_table_layout,select_gene,select_trenchid,select_unpacked_trenchid = tr.linked_gene_table(gene_table_out,index_key='Gene',trenchids_as_list=True,trenchid_column='phenotype trenchids')

In [ ]:
gene_table_layout

In [ ]:
output_display,save_button = tr.linked_kymograph_for_gene_table(kymo_xarr,wrapped_kymo_xarr,gene_table_out,select_gene,select_trenchid,select_unpacked_trenchid=select_unpacked_trenchid,\
                                                    trenchid_column='phenotype trenchids',y_scale=1.2,x_window_size=1200)

In [ ]:
output_display

In [ ]:
dask_controller.shutdown()